# Getting many lightcurvces

First, Import stuff we'll need and instantiate a FASTDB.  (This assumes you've set yourself up as described in the [FASTDB docs](https://fastdb.readthedocs.io/en/latest/usage.html).



In [1]:
import sys
import time

import numpy
import pandas
import pyarrow


from fastdb_client import FASTDBClient
fdb = FASTDBClient( "production" )

## Getting lightcurves for a known set of objects

For the first example, let's assume that, somehow, you've come up with a list of objects within the projcessing version that you want to get the lightcurves for.  Right here, I'm hardcoding that list; see the "searching_for_objects_and_sources.ipynb" notebook for examples in generating this list.

Note that I'm using the FASTDB-defined "root ids" here, not the LSST-defined "diaobjectid".  In general, when interacting with FASTDB, you're better off using the root ids.  The reason is that the LSST diaobjectids are not unique.  That is, a given diaobjectid (as far as I can tell) *does* always refer to the same actual transient, but the same transient will sometimes have more than one diaobjectid even within a single processing version, and, what's more, a given diasource will be associated with different of these degenerate diaobjectids in different alets.  FASTDB deduplicates the diaobjectid by grouping everything within 1" of each other under the same root id.  Furthermore, whereas LSST will be assigning new diaobjectid values to each actual transient with each data release, rootid is universal, and the same root id will be associated with all of the diabojectids in all of the processing versions.


In [2]:
rootids = [ '00a6f58b-158f-4eec-a603-3e01d8b8c498',
            '175bf3ce-5cf1-4efd-9c6a-57b0b8bfe464',
            '2b7e79eb-45a0-4149-a002-4127439917ef',
            '3fc12ba3-e3ee-49c1-91ed-47e3aea52060',
            '5f9bca57-e733-476e-9d85-45fe5a0c7f12',
            '6fe30896-8027-40e5-ba60-c769191a6e97',
            '99d76541-c3ce-4c05-ab1b-ecb9eaf0c9eb',
            'afd33b78-43fd-4edc-bf79-a1e21fa15d1e',
            'c509e1ac-32c3-498d-81cc-c1ca3cc5011b',
            'd6329978-1b61-4aa1-80f7-09ce497155b1' ]



To actually get the lightcurve, just hit the [ltcv/getmanyltcvs](https://fastdb.readthedocs.io/en/latest/usage.html#ltcv-getmanyltcvs) API endpoint.

**Don't pass too many objects.**   If you do, the query will take a long time, and if it takes long enough, the connection will time out before you get your answer.  Hundreds of items should easily be safe.  Thousands of items is probably safe.  Tens of thousands is getting borderline.

In [3]:
manyltcvs = fdb.post( "ltcv/getmanyltcvs/realtime", json={'objids': rootids} )

print( f"Got back a {type(manyltcvs)} with {len(manyltcvs)} elements." )

Got back a <class 'list'> with 10 elements.


You get back a list with each element being a lightcurve.  It's possible that you will get fewer than you asked for.  There are two situations in which this could happen.  The first is if some of the objects you asked for don't have any photometry in the processing version you specified.  The second is if you specified `diaobjectid` values instead of `rootid` values, and some of the `diaobjectid` values on your list were the same actual transient as others.

Each element of the list is a dictionary:

In [4]:
print( f"manyltcvs[0].keys() = {manyltcvs[0].keys()}" )

manyltcvs[0].keys() = dict_keys(['rootid', 'diaforcedsourceid', 'diasourceid', 'forced_diaobjectid', 'source_diaobjectid', 'visit', 'mjd', 'band', 'flux', 'fluxerr', 'isdet', 'ispatch'])


In this dictionary, `rootid` is just a single value, the rootid for which this is the lightcurve.  The rest of these are lists, all of the same length.  They should all be sorted by mjd.  The coumns are:

* `mjd` : MJD (from the LSST alert field `midpointMJDTai`)
* `band` : The filter (single-character, one of `u`, `b`, `g`, `r`, `i`, `z`, or `y`).  (I *think* the `y` is lowercase.)
* `flux` : Flux in nJy (from the LSST alert field `psfFlux`)
* `fluxerr` : uncertainty on flux (from the LSST alert field `psfFluxErr`)
* `visit` : The visit number
* `diasourceid` : The LSST-provided id for this *detection*.  For the `realtime` processing version, it will have come *either* from the `diasourceid` field at the top level of an alert we ingested, or from a `prvDiaSources` array in an alert we ingested from a broker.  This will be None if we don't have a record of a dection, *only* forced photometry for thi spoint.
* `diaforcedsourceid` : The LSST-provided id for this *forced photometry point*.  For the `realtime` processingt version, it will have come from the `prvDiaForcedSources` array in an alert we ingested from a broker.  This will be None if we don't have forced photometry for this point, only a detection.
* `source_diaobjectid` : The `diaobjectid` that FASTDB associates with this `diasourceid`.  This is *not* necessarily the same as what will be in a future putative PPDB put out by LSST, because empirically the same `diasourceid` has *different* `diaobjectid` values in different alerts.  (Alerts generated for that source will presumably all have the same `diaobjectid` — in fact, normally, I'd expect LSST only ever to generate a single alert for a single `diasourceid`, though we may have received it multiple times from different brokers.  However, when this same `diasourceid` shows up in the `prvDiaSources` array in a later alert, it may have a different `diaobjectid` attached to it.  FASTDB just associates the first one it sees.  This is part of why we encourage you to use `rootid` whenever possible when itneracting with FASTDB.)  This will be None if `diasourceid` is None.
* `forced_diaobjectid` : The `diaobjectid` that FASTDB associates with this `diaforcedsourcid`.  All the same caveates that apply to `source_diaobjectid` apply here.
* `isdet` : True if, as far as FASTDB knows, this object was detected by LSST in a transient search on a difference image.  (FASTDB only knows what it has ingested from alerts, so it's *possible* that LSST has a `diasourceid` for a point that never made it into any of the alerts that got to FASTDB.)  False if this point has no `diasourceid` associated with it.  (`isdet == (diasourceid is not None)` should always be True; if it's not, it indicates a logic error somewhere in the FASTDB code.)
* `ispatch` : True if `flux` and `fluxerr` come from the original detection in LSST's search, False if `flux` and `fluxerr` come from forced photometry.

There are some additional parameters you can give in the `json=` dictionary to `fdb.post("ltcv/getmanyltcvs...`.  One is `which`, which can be `detections`, `forced`, or `patch`.  The default is `patch`.  These mean:
* `detections` : only return detections.  (I *think* that the `...forced...` columns will be omitted in the return in this case.)  The `flux` and `fluxerr` values come from the *detections*, never from the forced photomery, in this case.
* `forced` : only return forced photometry.
* `patch` (default) : return forced photometry where we have it.  Where we don't, "patch" in the photometry from the detection.  This gives the most complete lightcurve, and is usually what you want.

If you're thinking about doing precision cosmology with these lightcurves... don't.  The "patch" version should scare you, because the data is heterogeneous.  You *might* think that just asking for "forced" will give you something consistent, but **be very wary of that**.  The forced photometry we save is from the *first alert* we ingest that had a given `diaforcedsourceid`.  I don't know if LSST ever redoes forced photometry, improving the position as more information is available.  If they do, we don't know about it.  Presumably once data releases come out, it will be all done consistently and correct, but for the alert stream (the `realtime` processing version in FASTDB), everything should be a little but suspect.  These lightcurves should be fine for identification and follow-up, but there no real hope of truly understanding detailed errors and correlations in them.  **Do not do precision cosmology with the alert stream, wait until we have an LSST data release.**

In [5]:
for ltcv in manyltcvs:
    print( f"{ltcv['rootid']} has {len(ltcv['mjd'])} mjd values and {len(ltcv['flux'])} flux values." )

00a6f58b-158f-4eec-a603-3e01d8b8c498 has 374 mjd values and 374 flux values.
175bf3ce-5cf1-4efd-9c6a-57b0b8bfe464 has 158 mjd values and 158 flux values.
2b7e79eb-45a0-4149-a002-4127439917ef has 1232 mjd values and 1232 flux values.
3fc12ba3-e3ee-49c1-91ed-47e3aea52060 has 568 mjd values and 568 flux values.
5f9bca57-e733-476e-9d85-45fe5a0c7f12 has 215 mjd values and 215 flux values.
6fe30896-8027-40e5-ba60-c769191a6e97 has 520 mjd values and 520 flux values.
99d76541-c3ce-4c05-ab1b-ecb9eaf0c9eb has 1217 mjd values and 1217 flux values.
afd33b78-43fd-4edc-bf79-a1e21fa15d1e has 165 mjd values and 165 flux values.
c509e1ac-32c3-498d-81cc-c1ca3cc5011b has 237 mjd values and 237 flux values.
d6329978-1b61-4aa1-80f7-09ce497155b1 has 140 mjd values and 140 flux values.


If you want to put these into a pandas dataframe, **do not use pandas.DataFrame()**.  Reason: the `diaforcedsourceid`, `diasourceid`, `forced_diaobjectid`, and `source_diaobjectid` columnns are all a mixture of 64-bit intetegers and None.  (See above for why they might be None.)  Pandas, by default, can't handle a column with a mix of integers and None, so it converts them to 64-bit floats and uses NaN in place of None.  Unfortunately, 64-bit floats only have 53 bits of numerical precision, so the 64-bit integer id will be modified by this conversion if it's ≳2⁵² (which often happens!).  For floats, you usually don't care; that's 15 digits of precision, which for any physical measurement any astronomer ever talks about is overkill.  (But, insert concerns about floating point overflow and repeated operations....)  For integer IDs, the *exact value* matters.

There is a solution: Pyarrow *does* have a datatype that can conbine integers and Nones, and Pandas can use those datatypes.  Alas, there is no way to tell Pandas "please use all Pyarrow types" when making a dataframe, you'd have to build a bunch of Serieses one by one and paste them into a dataframe.  (Cf: https://github.com/LSSTDESC/FASTDB/blob/16ece30fdd246d87ef90f9fc93bc4177d9946c28/src/util.py#L424 ).  However, you *can* make a Pyarrow table and then convert it to Pandas.  As such, I recommend you always use this pattern:

In [6]:
ltcv0 = manyltcvs[0].copy()
# Remove rootid, we don't need that in our dataframe as it's the same for all rows for one transient
del ltcv0['rootid']
df0 = pyarrow.Table.from_pydict( ltcv0 ).to_pandas( types_mapper=pandas.ArrowDtype )
df0

,diaforcedsourceid,diasourceid,forced_diaobjectid,source_diaobjectid,visit,mjd,band,flux,fluxerr,isdet,ispatch
0,<NA>,313664312980275280,<NA>,313664312980275280,2025110100252,60981.277865,z,-4559.175,650.7697,True,True
1,<NA>,313972153394397349,<NA>,313664312980275280,2026011000082,61051.078218,g,1582.9336,130.79808,True,True
2,170019696400138248,<NA>,313664312980275280,<NA>,2026021600107,61088.098476,r,1773.5338,201.1709,False,False
3,170028485907578933,170028485907578968,313664312980275280,313664312980275280,2026021800058,61090.048455,i,-2452.7327,369.911,True,False
4,170028486191218756,170028486191218829,313664312980275280,313664312980275280,2026021800060,61090.053544,i,-1488.4519,369.5259,True,False
...,...,...,...,...,...,...,...,...,...,...,...
369,170063687742128265,<NA>,313664312980275280,<NA>,2026022600188,61098.097396,r,1279.8419,332.82602,False,False
370,170063687995883564,170063687995883638,313664312980275280,313664312980275280,2026022600190,61098.098271,r,1957.7129,323.3472,True,False
371,170063688277950528,<NA>,313664312980275280,<NA>,2026022600192,61098.099141,r,1619.3119,314.48135,False,False
372,170063688547959085,<NA>,313664312980275280,<NA>,2026022600194,61098.100096,r,1472.6973,314.90063,False,False


### Avoiding Nones in ids

There is a workaround for the columns which are a mixture of ints and Nones.  It's inelegant, but many of us have used it for many years.  (I have memories of doing this in the 90s.)  (The 1990s, I'm not *that* old.)  (But I am pretty sure Ada Lovelace used this trick when she wrote out FITS tables.)  *If* we believe that the ID columns are all, technically, signed integers (I think that is the datatype in the LSST alerts.), but that meaningful ones will only ever be positive, we can replace all the None values with a negative number.  In that case, it is safe to go straigth to a Pandas dataframe.  Here's how to tell FASTDB to do that:



In [7]:
manyltcvs = fdb.post( "ltcv/getmanyltcvs/realtime", json={'objids': rootids, 'nonevalue': -999} )
df0 = pandas.DataFrame( manyltcvs[0] )
df0

,rootid,diaforcedsourceid,diasourceid,forced_diaobjectid,source_diaobjectid,visit,mjd,band,flux,fluxerr,isdet,ispatch
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,-999,313664312980275280,-999,313664312980275280,2025110100252,60981.277865,z,-4559.1750,650.76970,True,True
1,00a6f58b-158f-4eec-a603-3e01d8b8c498,-999,313972153394397349,-999,313664312980275280,2026011000082,61051.078218,g,1582.9336,130.79808,True,True
2,00a6f58b-158f-4eec-a603-3e01d8b8c498,170019696400138248,-999,313664312980275280,-999,2026021600107,61088.098476,r,1773.5338,201.17090,False,False
3,00a6f58b-158f-4eec-a603-3e01d8b8c498,170028485907578933,170028485907578968,313664312980275280,313664312980275280,2026021800058,61090.048455,i,-2452.7327,369.91100,True,False
4,00a6f58b-158f-4eec-a603-3e01d8b8c498,170028486191218756,170028486191218829,313664312980275280,313664312980275280,2026021800060,61090.053544,i,-1488.4519,369.52590,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
369,00a6f58b-158f-4eec-a603-3e01d8b8c498,170063687742128265,-999,313664312980275280,-999,2026022600188,61098.097396,r,1279.8419,332.82602,False,False
370,00a6f58b-158f-4eec-a603-3e01d8b8c498,170063687995883564,170063687995883638,313664312980275280,313664312980275280,2026022600190,61098.098271,r,1957.7129,323.34720,True,False
371,00a6f58b-158f-4eec-a603-3e01d8b8c498,170063688277950528,-999,313664312980275280,-999,2026022600192,61098.099141,r,1619.3119,314.48135,False,False
372,00a6f58b-158f-4eec-a603-3e01d8b8c498,170063688547959085,-999,313664312980275280,-999,2026022600194,61098.100096,r,1472.6973,314.90063,False,False


In fact, at this point, I think you could feed the entire return to `pandas.DataFrame()`:

In [8]:
monsterdf = pandas.DataFrame( manyltcvs )
monsterdf

,rootid,diaforcedsourceid,diasourceid,forced_diaobjectid,source_diaobjectid,visit,mjd,band,flux,fluxerr,isdet,ispatch
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,"[-999, -999, 170019696400138248, 1700284859075...","[313664312980275280, 313972153394397349, -999,...","[-999, -999, 313664312980275280, 3136643129802...","[313664312980275280, 313664312980275280, -999,...","[2025110100252, 2026011000082, 2026021600107, ...","[60981.277865211654, 61051.07821800333, 61088....","[z, g, r, i, i, i, i, i, i, i, i, i, i, g, g, ...","[-4559.175, 1582.9336, 1773.5338, -2452.7327, ...","[650.7697, 130.79808, 201.1709, 369.911, 369.5...","[True, True, False, True, True, True, False, F...","[True, True, False, False, False, False, False..."
1,175bf3ce-5cf1-4efd-9c6a-57b0b8bfe464,"[-999, -999, 170032916246560770, 1700329171614...","[170028528400072705, 170028534452977685, 17003...","[-999, -999, 170028534452977685, 1700285344529...","[170028534452977685, 170028534452977685, 17002...","[2026021800374, 2026021800419, 2026021900298, ...","[61090.25448737413, 61090.31719297204, 61091.2...","[i, i, i, i, i, i, g, g, g, g, g, r, r, r, r, ...","[2212.2239, 2265.172, 1656.1083, 2465.2441, 14...","[364.18674, 337.34653, 374.9632, 323.3789, 420...","[True, True, True, True, False, False, False, ...","[True, True, False, False, False, False, False..."
2,2b7e79eb-45a0-4149-a002-4127439917ef,"[-999, -999, -999, -999, -999, -999, -999, -99...","[313853517449658448, 313853517582827626, 31385...","[-999, -999, -999, -999, -999, -999, -999, -99...","[313853517449658448, 313853517449658448, 31385...","[2025121400911, 2025121400912, 2025121400913, ...","[61024.24593270644, 61024.24643140827, 61024.2...","[i, i, i, i, i, i, i, i, i, i, i, i, i, i, i, ...","[3421.0386, 3274.8313, 3239.2966, 3470.1326, 3...","[366.00363, 366.67087, 366.73062, 384.36075, 3...","[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, True, Tru..."
3,3fc12ba3-e3ee-49c1-91ed-47e3aea52060,"[-999, -999, -999, -999, -999, -999, 170028527...","[170028526469644320, 170028526603337775, 17002...","[-999, -999, -999, -999, -999, -999, 170028527...","[170028527022768199, 170028527022768199, 17002...","[2026021800360, 2026021800361, 2026021800364, ...","[61090.247937232445, 61090.2484597937, 61090.2...","[i, i, i, i, i, i, i, i, i, i, i, g, g, g, g, ...","[1829.2694, 1845.1823, 2108.008, 1705.1083, 19...","[288.94708, 308.54672, 305.25497, 304.49792, 2...","[True, True, True, True, True, True, True, Tru...","[True, True, True, True, True, True, False, Fa..."
4,5f9bca57-e733-476e-9d85-45fe5a0c7f12,"[-999, -999, -999, -999, -999, -999, 170019696...","[313761042327929211, 313897383949238527, 31397...","[-999, -999, -999, -999, -999, -999, 313761042...","[313761042327929211, 313761042327929211, 31376...","[2025112300046, 2025122400062, 2026011000082, ...","[61003.324393250674, 61034.16130408224, 61051....","[g, g, g, g, g, g, r, i, i, i, i, i, i, g, g, ...","[771.05927, 840.9975, 1199.3889, 1271.8796, 11...","[156.66542, 121.78, 135.36357, 150.22276, 160....","[True, True, True, True, True, True, False, Tr...","[True, True, True, True, True, True, False, Fa..."
5,6fe30896-8027-40e5-ba60-c769191a6e97,"[-999, -999, -999, -999, -999, 170028527115042...","[170028526443431304, 170028526576075487, 17002...","[-999, -999, -999, -999, -999, 170028526443431...","[170028526443431304, 170028526443431304, 17002...","[2026021800360, 2026021800361, 2026021800362, ...","[61090.247937232445, 61090.2484597937, 61090.2...","[i, i, i, i, i, i, i, i, i, i, i, i, i, g, g, ...","[2771.8381, 3635.8796, 2793.5671, 3142.2166, 3...","[356.72998, 368.0168, 380.12418, 349.7459, 359...","[True, True, True, True, True, True, False, Tr...","[True, True, True, True, True, False, False, F..."
6,99d76541-c3ce-4c05-ab1b-ecb9eaf0c9eb,"[-999, -999, -999, -999, -999, -999, -999, -99...","[313853517492650005, 313853517625294876, 31385...","[-999, -999, -999, -999, -999, -999, -999, -99...","[313853517492650005,

Well, OK, that's not going to be convenient (uness you use something like "nested pandas"), but you can fix this with:

In [9]:
cols = [ c for c in monsterdf.columns if c != 'rootid' ]
monsterdf = monsterdf.explode( cols )
monsterdf

,rootid,diaforcedsourceid,diasourceid,forced_diaobjectid,source_diaobjectid,visit,mjd,band,flux,fluxerr,isdet,ispatch
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,-999,313664312980275280,-999,313664312980275280,2025110100252,60981.277865,z,-4559.175,650.7697,True,True
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,-999,313972153394397349,-999,313664312980275280,2026011000082,61051.078218,g,1582.9336,130.79808,True,True
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,170019696400138248,-999,313664312980275280,-999,2026021600107,61088.098476,r,1773.5338,201.1709,False,False
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,170028485907578933,170028485907578968,313664312980275280,313664312980275280,2026021800058,61090.048455,i,-2452.7327,369.911,True,False
0,00a6f58b-158f-4eec-a603-3e01d8b8c498,170028486191218756,170028486191218829,313664312980275280,313664312980275280,2026021800060,61090.053544,i,-1488.4519,369.5259,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
9,d6329978-1b61-4aa1-80f7-09ce497155b1,170063691137417487,-999,170050472143487126,-999,2026022600213,61098.116189,z,1879.2568,527.52686,False,False
9,d6329978-1b61-4aa1-80f7-09ce497155b1,170068070214664450,-999,170050472143487126,-999,2026022700072,61099.04384,i,1572.01,314.07968,False,False
9,d6329978-1b61-4aa1-80f7-09ce497155b1,170076861495771259,-999,170050472143487126,-999,2026030100036,61101.030547,i,2350.5945,333.3394,False,False
9,d6329978-1b61-4aa1-80f7-09ce497155b1,170103252217495735,-999,170050472143487126,-999,2026030700054,61107.030626,i,3708.2332,464.7799,False,False


You can of course play other games, e.g.:

In [10]:
monsterdf.sort_values( ['rootid', 'band', 'mjd'] ).set_index( ['rootid', 'band', 'mjd'] )

diaforcedsourceid  \
rootid                               band mjd                                
00a6f58b-158f-4eec-a603-3e01d8b8c498 g    61051.078218                -999   
                                          61090.062787  170028488595079193   
                                          61090.063657  170028488870330419   
                                          61090.064521  170028489154494484   
                                          61090.065383  170028489422405702   
...                                                                    ...   
d6329978-1b61-4aa1-80f7-09ce497155b1 z    61098.072481  170063680617578627   
                                          61098.073356  170063680890208487   
                                          61098.074230  170063681153401018   
                                          61098.082121  170063683598155941   
                                          61098.116189  170063691137417487   

                                                               diasourceid  \
rootid                               band mjd                                
00a6f58b-158f-4eec-a603-3e01d8b8c498 g    61051.078218  313972153394397349   
                                          61090.062787                -999   
                                          61090.063657                -999   
                                          61090.064521                -999   
                                          61090.065383                -999   
...                                                                    ...   
d6329978-1b61-4aa1-80f7-09ce497155b1 z    61098.072481                -999   
                                          61098.073356                -999   
                                          61098.074230                -999   
                                          61098.082121                -999   
                                          61098.116189                -999   

                                                        forced_diaobjectid  \
rootid                               band mjd                                
00a6f58b-158f-4eec-a603-3e01d8b8c498 g    61051.078218                -999   
                                          61090.062787  313664312980275280   
                                          61090.063657  313664312980275280   
                                          61090.064521  313664312980275280   
                                          61090.065383  313664312980275280   
...                                                                    ...   
d6329978-1b61-4aa1-80f7-09ce497155b1 z    61098.072481  170050472143487126   
                                          61098.073356  170050472143487126   
                                          61098.074230  170050472143487126   
                                          61098.082121  170050472143487126   
                                          61098.116189  170050472143487126   

                                                        source_diaobjectid  \
rootid                               band mjd                                
00a6f58b-158f-4eec-a603-3e01d8b8c498 g    61051.078218  313664312980275280   
                                          61090.062787                -999   
                                          61090.063657                -999   
                                          61090.064521                -999   
                                          61090.065383                -999   
...                                                                    ...   
d6329978-1b61-4aa1-80f7-09ce497155b1 z    61098.072481                -999   
                                          61098.073356                -999   
                                          61098.074230                -999   
                                          61098.082121                -999   
                                          61098.116189                -999  

## Getting all the lightcurves

As a second use case, you might want *all* of the lightcurves in a given processing version.  Right now, there are only ~60k lightcurves in the database, so you probably could get them all at once, but eventually that won't scale.  If you don't specify a list of objects you want lightcurves for, then you can give the `limit` and `offset` parameters to grab a subset.  Feel free to play with the size of the subset you ask for, but probably ~10,000 is the limit (and may itself be too much).  (That will depend on the size of the lightcurves.  Right now, the database leans heavily towards DDF transients that have a *lot* of points.)

(Obligatory warning: the `realtime` processing version is ingesting new sources all the time, as it gets new alerts from brokers.  This means that from one call to the next, the total list of lightcurves might change, so your `limit` and `offset` may get you a duplicate, or may miss something.  This will not be a problem for data releases (usually), so just be aware that the realtime processing version is what it is.)

In [11]:
t0 = time.perf_counter()
manyltcvs = fdb.post( "ltcv/getmanyltcvs/realtime", json={'limit': 1000, 'nonevalue': -999} )
sys.stderr.write( f"Got {len(manyltcvs)} lightcurves in {time.perf_counter()-t0:.2f} sec.\n" )

# Notice that these come sorted by rootid:
sys.stderr.write( f"First rootid: {manyltcvs[0]['rootid']}, last rootid: {manyltcvs[-1]['rootid']}\n" )

# Get the next batch
t0 = time.perf_counter()
manyltcvs = fdb.post( "ltcv/getmanyltcvs/realtime", json={'limit': 1000, 'offset': 1000, 'nonevalue': -999} )
sys.stderr.write( f"Got the next {len(manyltcvs)} lightcurves in {time.perf_counter()-t0:.2f} sec.\n" )
sys.stderr.write( f"In the second batch, first rootid: {manyltcvs[0]['rootid']}, last rootid: {manyltcvs[-1]['rootid']}\n" )

Got 1000 lightcurves in 10.36 sec.
First rootid: 00013902-e4e5-4b4a-995b-3a5c36f38d15, last rootid: 03c7a3d6-f08c-41cd-9bed-704adfc0d727
Got the next 1000 lightcurves in 9.69 sec.
In the second batch, first rootid: 03c869d4-0b1e-4691-9913-855fa48b12a2, last rootid: 07804a30-e0f8-4fcc-8256-a1740193c272


123